In [4]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
import json
import re
import os
from unicodedata import normalize
#from .unipath_gdg import UNIPATH_GDG

In [5]:
def preprocess_chunks(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    documents = []
    for chunk in data:
        text = chunk['text']
        # Clean excessive whitespace/newlines
        text = re.sub(r'\s+', ' ', text).strip()
        if len(text) > 50:  # Filter meaningful chunks
            documents.append({
                'text': text,
                'metadata': chunk.get('metadata', {}),
                'chunk_id': chunk.get('chunk_id')
            })
    return documents

In [6]:
# Example usage
documents = preprocess_chunks(r"D:\work\projects\UNIPATH_GDG\data\rag_dataset_fixed.json")
df = pd.DataFrame(documents)


In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Initialize Arabic/English bilingual model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Generate embeddings
embeddings = model.encode([doc['text'] for doc in documents])

# Create FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\EL Zahar\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\EL Zahar\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Example usage


In [9]:
def retrieve(query, k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k)
    
    results = []
    for idx in indices[0]:
        results.append({
            'text': documents[idx]['text'],
            'metadata': documents[idx]['metadata'],
            'score': 1 - distances[0][list(indices[0]).index(idx)]/2  # cosine similarity approx
        })
    return results

In [10]:
def rag_query(query, llm_client):
    # Retrieve relevant chunks
    context_chunks = retrieve(query, k=3)
    context = "\n\n".join([f"Document {i+1}: {c['text']}" for i, c in enumerate(context_chunks)])
    
    # Construct prompt with academic regulations context
    prompt = f"""You are an academic advisor for Helwan University Faculty of Science.
Answer based ONLY on the provided context. If information is not in context, say "Not specified in regulations."

Context:
{context}

Question: {query}

Answer in formal academic Arabic/English as appropriate:"""
    
    response = llm_client.generate(prompt)
    return {
        'answer': response,
        'sources': [c['metadata'].get('source', 'N/A') for c in context_chunks]
    }

In [12]:
query = "ما هي شروط التخرج من قسم الرياضيات؟"
results = retrieve(query)

print(f"نتائج البحث عن: '{query}'\n")
for i, res in enumerate(results, 1):
    print(f"{i}. score: {res['score']:.3f}")
    print(f"   text: {res['text'][:300]}...\n")

نتائج البحث عن: 'ما هي شروط التخرج من قسم الرياضيات؟'

1. score: -6.063
   text: غالبا 3 متطلبات 60 30 90 66,18 يبدأ الطالب دراستها اعتبارا التخصص المنفر د من المستوى الثانى 4 متطلبات 46 20 66 48,53 يبدأ الطالب دراستها اعتبارا التخصص الرئيسى من المستوى الثانى 5 متطلبات 16 8 24 17,65 يبدأ الطالب دراستها اعتبارا التخصص الفرعى من المستوى الثانى 6 متطلبات 60 30 90 66,18 يبدأ الطالب ...

2. score: -6.086
   text: and or demonstrated as/ junior level work in the sciences. Topics This course is designed to develop a basic competence in many areas of Mat3269 Mathematics for Science engineering are included. applications in applied mathematics, physics, chemistry, biology/189 - procedural) and debugging, plottin...

3. score: -6.835
   text: للطالب المتفوق الحاصل على تقدير ممتاز 3.667) (A على األقل بعد المستوى األول - أن يسجل 20 ساعة معتمدة فى الفصلالدراسى الواحد وبحد أقصى 4 مرات طوال فترة. الدراسة )ب( يجوز لمجلس الكلية زيادة الحد األقصى للعبء الدراسى إلى 22 ساعة معتمدة لمرة واحدة للطالب الذى إ